# 04 · Route each complaint to the right team (text classification)

**The business problem:** a complaint arrives as free text. Someone has to read it and send it to the right team:
cards, mortgages, debt collection and so on. Here a model reads the customer's words and predicts the **product**.

**How we judge it:** the customer also picked a product on the form, so we know the right answer for every
complaint. We hide 20% of complaints, train on the rest, and check the predictions on the hidden 20%.

**Steps:** a "guess the biggest class" baseline first, then three classic text models. Every run is logged in
MLflow so the comparison is saved, not just printed.

In [1]:
from radar import setup, run_sql, save_table, mlflow_setup, ON_DATABRICKS, CATALOG, SCHEMA
setup()

'Local: tables go to data/complaint_radar.duckdb'

### Load the complaints

In [2]:
data = run_sql("SELECT complaint_id, product, text, narrative FROM complaints")
data["product"].value_counts()

product
Credit card         3912
Money transfer      3890
Bank account        3882
Debt collection     3844
Credit reporting    3768
Mortgage            2186
Vehicle loan        1691
Student loan        1415
Personal loan       1145
Prepaid card         510
Debt management      315
Name: count, dtype: int64

### Split into training and test sets
`stratify` keeps the same mix of products in both parts. `random_state` makes the split the same every run.

In [3]:
from sklearn.model_selection import train_test_split

train, test = train_test_split(data, test_size=0.2, stratify=data["product"], random_state=42)
print(f"train: {len(train):,} complaints · test: {len(test):,} complaints")

train: 21,246 complaints · test: 5,312 complaints


### Baseline: always guess the most common product
Any real model has to beat this. With a balanced sample it's low, which is the point: it shows how much the
model actually learns from the words.

In [4]:
from sklearn.metrics import accuracy_score, f1_score

most_common = train["product"].mode()[0]
baseline = {"accuracy": accuracy_score(test["product"], [most_common] * len(test)),
            "macro_f1": f1_score(test["product"], [most_common] * len(test), average="macro")}
baseline

{'accuracy': 0.14740210843373494, 'macro_f1': 0.02335744649116265}

### Turn words into numbers: TF-IDF
A model can't read text, so each complaint becomes a long list of numbers, one per word or two-word phrase.
**TF-IDF** gives a phrase a high number when it appears often in this complaint but rarely in others,
so "escrow" counts for a lot and "the" counts for nothing.

In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer

def tfidf():
    return TfidfVectorizer(ngram_range=(1, 2),     # single words and two-word phrases ("late fee")
                           min_df=3,               # ignore phrases seen in fewer than 3 complaints
                           max_df=0.5,             # ignore phrases in more than half of all complaints
                           max_features=150_000,
                           sublinear_tf=True,      # 10 mentions isn't 10 times as important as 1
                           stop_words="english")

### Three models to compare
* **Logistic regression**: learns a weight for every phrase and product; easy to explain.
* **Linear SVM**: similar, often slightly more accurate on text.
* **Complement naive Bayes**: very fast, a classic for text.

In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import ComplementNB
from sklearn.svm import LinearSVC
from sklearn.pipeline import make_pipeline

MODELS = {
    "logistic_regression": lambda: LogisticRegression(C=4.0, max_iter=2000),
    "linear_svm": lambda: LinearSVC(C=0.5),
    "complement_nb": lambda: ComplementNB(alpha=0.3),
}

### Train each model and log it in MLflow
For each model: train, predict the test set, and log the settings, the scores and the training time.

In [7]:
import time
mlflow = mlflow_setup()

results, fitted = [], {}
for name, make in MODELS.items():
    with mlflow.start_run(run_name=name):
        model = make_pipeline(tfidf(), make())
        start = time.time()
        model.fit(train["text"], train["product"])
        seconds = time.time() - start
        pred = model.predict(test["text"])
        scores = {"accuracy": accuracy_score(test["product"], pred),
                  "macro_f1": f1_score(test["product"], pred, average="macro"),
                  "train_seconds": seconds}
        mlflow.log_params({"model": name, "train_rows": len(train), "test_rows": len(test)})
        mlflow.log_metrics(scores)
        mlflow.log_metric("baseline_macro_f1", baseline["macro_f1"])
        results.append({"model": name, **scores})
        fitted[name] = model

import pandas as pd
data =pd.DataFrame(results).sort_values("macro_f1", ascending=False).round(4)
data

/opt/hostedtoolcache/Python/3.12.16/x64/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2026/11/15 18:08:20 INFO mlflow.store.db.utils: Creating initial MLflow database tables...


2026/11/15 18:08:20 INFO mlflow.store.db.utils: Updating database tables


2026/11/15 18:08:21 INFO mlflow.tracking.fluent: Experiment with name 'complaint-radar' does not exist. Creating a new experiment.


,model,accuracy,macro_f1,train_seconds
1,linear_svm,0.8276,0.7636,7.0039
0,logistic_regression,0.8259,0.7501,32.5116
2,complement_nb,0.7848,0.6826,5.5598


### Pick the best model
We pick by **macro F1**: the F1 score of every product, averaged with equal weight. Plain accuracy would let the
model ignore small products and still look good.

In [8]:
best_name = data.iloc[0]["model"]
best = fitted[best_name]
pred = best.predict(test["text"])
print("best:", best_name)

best: linear_svm


### Scores per product
Precision: when the model says "Mortgage", how often it's right. Recall: of all real mortgage complaints,
how many it found.

In [9]:
from sklearn.metrics import classification_report
data = pd.DataFrame(classification_report(test["product"], pred, output_dict=True)).T.round(3)
save_table(data.rename_axis("label").reset_index(), "class_report")
data

,precision,recall,f1-score,support
Bank account,0.760,0.832,0.795,776.000
Credit card,0.758,0.839,0.796,783.000
Credit reporting,0.836,0.886,0.860,754.000
Debt collection,0.826,0.827,0.827,769.000
Debt management,0.882,0.238,0.375,63.000
Money transfer,0.872,0.847,0.859,778.000
Mortgage,0.930,0.943,0.936,437.000
Personal loan,0.733,0.515,0.605,229.000
Prepaid card,0.843,0.422,0.562,102.000
Student loan,0.930,0.933,0.931,283.000


### Confusion matrix: which products get mixed up
Each row is the real product; each column is what the model said. The diagonal is "right". The biggest
off-diagonal cells show where the categories genuinely overlap: a complaint about a collector reporting a debt
to the credit bureaus is both "Debt collection" and "Credit reporting".

In [10]:
from sklearn.metrics import confusion_matrix
labels = sorted(data.index.drop(["accuracy", "macro avg", "weighted avg"]))
matrix = confusion_matrix(test["product"], pred, labels=labels)
data = pd.DataFrame(matrix, index=labels, columns=labels)
save_table(data.rename_axis("actual").reset_index().melt(id_vars="actual", var_name="predicted", value_name="complaints"), "confusion")
data

,Bank account,Credit card,Credit reporting,Debt collection,Debt management,Money transfer,Mortgage,Personal loan,Prepaid card,Student loan,Vehicle loan
Bank account,646,51,6,4,0,60,4,2,3,0,0
Credit card,52,657,25,29,0,8,3,2,3,1,3
Credit reporting,3,22,668,46,0,1,4,1,0,7,2
Debt collection,7,32,62,636,1,1,2,10,0,8,10
Debt management,8,11,9,12,15,0,4,3,0,0,1
Money transfer,87,20,0,0,0,659,4,2,2,1,3
Mortgage,7,6,2,1,0,1,412,4,0,1,3
Personal loan,14,34,4,28,1,10,5,118,0,2,13
Prepaid card,22,18,0,1,0,16,0,2,43,0,0
Student loan,0,3,7,4,0,0,1,3,0,264,1


### How sure is the model? (for auto-routing)
In real life you wouldn't trust every prediction. A common setup: send a complaint automatically when the model
is confident, and to a person when it isn't. Logistic regression gives a probability for each product, so we
check how many complaints clear each confidence bar, and how accurate those are.

In [11]:
import numpy as np
lr = fitted["logistic_regression"]
proba = lr.predict_proba(test["text"])
confidence = proba.max(axis=1)
lr_pred = lr.classes_[proba.argmax(axis=1)]
rows = []
for bar in [0.0, 0.5, 0.6, 0.7, 0.8, 0.9, 0.95]:
    sure = confidence >= bar
    rows.append({"confidence_at_least": bar,
                 "auto_routed_pct": round(100 * sure.mean(), 1),
                 "accuracy_of_auto_routed_pct": round(100 * (lr_pred[sure] == test["product"].values[sure]).mean(), 1)})
data = pd.DataFrame(rows)
save_table(data, "routing")
data

,confidence_at_least,auto_routed_pct,accuracy_of_auto_routed_pct
0,0.00,100.0,82.6
1,0.50,83.4,89.1
2,0.60,75.6,91.5
3,0.70,67.3,93.3
4,0.80,57.7,94.8
5,0.90,42.7,97.2
6,0.95,31.7,98.2


### Words that push a complaint towards each product
Logistic regression is easy to explain: every phrase has a weight per product. The highest weights are the
phrases that most strongly point to that product.

In [12]:
vec, clf = lr.named_steps["tfidfvectorizer"], lr.named_steps["logisticregression"]
words = np.array(vec.get_feature_names_out())
rows = []
for i, product in enumerate(clf.classes_):
    top = np.argsort(clf.coef_[i])[::-1][:10]
    rows.append({"product": product, "top_phrases": ", ".join(words[top])})
data = pd.DataFrame(rows)
save_table(data, "top_phrases")
data

,product,top_phrases
0,Bank account,"bank, funds, checking, chase, deposit, savings..."
1,Credit card,"card, credit card, capital, citi, synchrony, c..."
2,Credit reporting,"report, equifax, experian, accounts, credit re..."
3,Debt collection,"debt, collection, collections, owe, owed, cred..."
4,Debt management,"debt, credit, settlement, program, charged, fe..."
5,Money transfer,"paypal, zelle, cashapp, money, funds, transfer..."
6,Mortgage,"mortgage, escrow, home, foreclosure, loan, pro..."
7,Personal loan,"loan, affirm, klarna, solar, advance, company,..."
8,Prepaid card,"card, gift, prepaid, money, gift card, funds, ..."
9,Student loan,"mohela, student, loans, loan, student loan, ne..."


### Look at mistakes
We save 12 random wrong predictions.
Reading wrong predictions is the fastest way to learn what the model can't do. Many "mistakes" are complaints
that really are about two products.

In [13]:
data = test.assign(predicted=pred)
data = data[data["product"] != data["predicted"]]
data = data.sample(min(12, len(data)), random_state=7)
data["excerpt"] = data["narrative"].str.slice(0, 280) + "…"
data = data[["complaint_id", "product", "predicted", "excerpt"]]
save_table(data, "mistakes")
data

,complaint_id,product,predicted,excerpt
16037,16221449,Student loan,Debt collection,I was a victim of identity theft I absolutely ...
19737,18264895,Money transfer,Bank account,I've been a member of cashapp for over XXXX an...
24008,21726424,Money transfer,Bank account,I am submitting this complaint regarding a ser...
25673,23439539,Debt collection,Personal loan,I originally agreed to pay XXXX XXXX XXXX abou...
22848,20823165,Credit card,Bank account,My account was closed without warning. I have ...
25970,23690530,Personal loan,Money transfer,placed an order using paypal pay in XXXX XXXX ...
15272,15848033,Debt collection,Credit reporting,A formal dispute under the Fair Credit Reporti...
25485,23228016,Mortgage,Personal loan,West Capital Lending received my information v...
5042,12077188,Personal loan,Money transfer,so what problem im having right now is im usin...
8217,13187445,Bank account,Prepaid card,XX/XX/year> I went to use my card and my accou...


### Save the predictions and the scores as tables
For the website and for anyone who wants to dig in.

In [14]:
data = test[["complaint_id", "product"]].assign(predicted=lr_pred, confidence=confidence.round(4))
save_table(data, "predictions")

'predictions: 5,312 rows'

In [15]:
data = pd.DataFrame(results).assign(baseline_macro_f1=baseline["macro_f1"], train_rows=len(train), test_rows=len(test))
save_table(data, "model_scores")

'model_scores: 3 rows'

### Register the model (Databricks only)
On Databricks the logistic regression is saved to Unity Catalog as `complaint_router`, so it can be loaded
by name or served behind an endpoint. Locally we skip this.

In [16]:
if ON_DATABRICKS:
    from mlflow.models import infer_signature
    with mlflow.start_run(run_name="register_complaint_router"):
        mlflow.sklearn.log_model(lr, name="model", input_example=test["text"].head(3).tolist(),
                                 signature=infer_signature(test["text"].head(20), lr.predict(test["text"].head(20))),
                                 registered_model_name=f"{CATALOG}.{SCHEMA}.complaint_router")
    print(f"registered {CATALOG}.{SCHEMA}.complaint_router")